# Craft Features from Linter Analysis
I need to create a dataset with the following features from the linter analysis to use them later to train a random forest model.
| Feature                  | Description                                                                 | Type    |
| :----------------------: | :-------------------------------------------------------------------------: | :-----: |
| Issue Code               |  Error/Warning Code e.g. PERF401                                            | String  |
| Row                      |  Line Position                                                              | Int     |
| Column                   |  Line Position                                                              | Int     |
| Fix Availability         |  Whether automatic fixes are available                                      | Boolean |
| Fix Applicability        |  Safe/Unsafe                                                                | String  |
| Fix Complexity           |  Number of needed edits (length of edits array)                             | Int     |
| Fix Description          |  Message Text e.g. “"Use a list comprehension to create a transformed list” | String  |
| Issue Frequency          |  Count of each issue type                                                   | Int     |
| Issues Per Line          |  Issues per line of code in a file                                          | Int     |
<!-- | Characters Per Line      |  Number of chars per line                                                   | Int     | -->


In [1]:
import ast
import glob
import json
import pandas as pd
from typing import List, Tuple
from sklearn.preprocessing import MultiLabelBinarizer

In [23]:
file_paths = glob.glob('lint_results/python/*.py.json')
file_paths[0]

'lint_results/python/SportsRank.py.json'

In [24]:
columns = ["title",
           "issue_code", 
           "line_row", 
           "line_column", 
           "fix_availability", 
           "fix_applicability", 
           "fix_complexity",
           "fix_description",
           "issue_frequency",
           "issues_per_line"]
linter_analysis_features = pd.DataFrame([], columns=columns)

In [25]:
for file_path in file_paths:
    with open(file_path, 'r') as file:
        linter_analysis_objects = json.load(file)
    title = file_path.replace('.py.json','').replace('lint_results/python/','')
    for linter_obj in linter_analysis_objects:
        issue_code = linter_obj['code']
        row = linter_obj['end_location']['row']
        column = linter_obj['end_location']['column'] 
        fix = linter_obj['fix']
        fix_availability = fix is not None
        fix_applicability = None
        fix_complexity = None
        if fix_availability:
            fix_applicability = fix['applicability']
            fix_complexity = len(fix['edits'])
            
        fix_description = linter_obj['message']
        issue_frequency = sum(obj['code'] == issue_code for obj in linter_analysis_objects)
        issues_per_line = sum(obj['end_location']['row'] == row for obj in linter_analysis_objects)
        linter_analysis_features.loc[len(linter_analysis_features)] = [
            title,
            issue_code,
            row,
            column,
            fix_availability,
            fix_applicability,
            fix_complexity,
            fix_description,
            issue_frequency,
            issues_per_line
        ]
linter_analysis_features.head()

,title,issue_code,line_row,line_column,fix_availability,fix_applicability,fix_complexity,fix_description,issue_frequency,issues_per_line
0,SportsRank,PLC0415,2,15,False,None,None,`import` should be at the top-level of a file,1,1
1,SportsRank,A001,3,10,False,None,None,Variable `input` is shadowing a Python builtin,1,1
2,SportsRank,E302,16,4,True,safe,1,"Expected 2 blank lines, found 0",4,2
3,SportsRank,ANN001,16,31,False,None,None,Missing type annotation for function argument ...,4,2
4,SportsRank,E302,25,4,True,safe,1,"Expected 2 blank lines, found 0",4,2


In [26]:
linter_analysis_features.to_csv('datasets/python_linter_analysis_features.csv', index=False)

### Features Per Line not Per Issue

In [27]:
crafted_features = pd.read_csv('datasets/python_linter_analysis_features.csv')

In [28]:
cleaned_crafted_features = crafted_features.drop(columns=['line_column', 'issue_frequency', 'fix_description'])

In [29]:
cleaned_crafted_features.dtypes

title                 object
issue_code            object
line_row               int64
fix_availability        bool
fix_applicability     object
fix_complexity       float64
issues_per_line        int64
dtype: object

In [30]:
grouped_crafted_features = cleaned_crafted_features.groupby(['title', 'line_row']).agg(
    any_fix_available=('fix_availability', 'any'),
    all_fixes_available=('fix_availability', 'all'),
    fix_availability_ratio=('fix_availability', 'mean'),

    unsafe_fixes=('fix_applicability', lambda x: (x == 'unsafe').sum()),
    safe_fixes=('fix_applicability', lambda x: (x == 'safe').sum()),

    avg_fix_complexity=('fix_complexity', 'mean'),
    max_fix_complexity=('fix_complexity', 'max'),
    min_fix_complexity=('fix_complexity', 'min'),
    sum_fix_complexity=('fix_complexity', 'sum'),

    unique_issue_types=('issue_code', pd.Series.nunique),
    issue_types_list=('issue_code', lambda x: list(set(x))),

    issues_per_line=('issues_per_line', 'first') 
).round(2).reset_index()

In [123]:
grouped_crafted_features.columns = [col for col in grouped_crafted_features.columns]
final_crafted_features = grouped_crafted_features.reset_index().drop('index', axis=1)
final_crafted_features.shape

(261, 14)

In [124]:
final_crafted_features.head()

,title,line_row,any_fix_available,all_fixes_available,fix_availability_ratio,unsafe_fixes,safe_fixes,avg_fix_complexity,max_fix_complexity,min_fix_complexity,sum_fix_complexity,unique_issue_types,issue_types_list,issues_per_line
0,A+BAgain?,7,True,True,1.0,0,1,1.0,1.0,1.0,1.0,1,[W292],1
1,A.WayTooLongWords,1,False,False,0.0,0,0,NaN,NaN,NaN,0.0,1,[ANN001],1
2,A.WayTooLongWords,2,False,False,0.0,0,0,NaN,NaN,NaN,0.0,1,[PLR2004],1
3,A.WayTooLongWords,7,True,True,1.0,1,0,2.0,2.0,2.0,2.0,1,[RET504],1
4,A.WayTooLongWords,8,True,True,1.0,0,1,1.0,1.0,1.0,1.0,1,[RET505],1


In [125]:
list(final_crafted_features.head()['issue_types_list'])

[['W292'], ['ANN001'], ['PLR2004'], ['RET504'], ['RET505']]

In [126]:
mlb = MultiLabelBinarizer()
encoded_issue_codes = mlb.fit_transform(final_crafted_features['issue_types_list'])
encoded_line_issues = pd.DataFrame(encoded_issue_codes, columns=mlb.classes_)
print(encoded_line_issues.shape)
final_crafted_features = final_crafted_features.drop(columns=['issue_types_list'])

(261, 46)


In [127]:
final_crafted_features = pd.concat([final_crafted_features, encoded_line_issues], axis=1, join='outer')
final_crafted_features.head()

,title,line_row,any_fix_available,all_fixes_available,fix_availability_ratio,unsafe_fixes,safe_fixes,avg_fix_complexity,max_fix_complexity,min_fix_complexity,...,PLW2901,Q000,RET504,RET505,RET507,RET508,RUF059,SIM110,W291,W292
0,A+BAgain?,7,True,True,1.0,0,1,1.0,1.0,1.0,...,0,0,0,0,0,0,0,0,0,1
1,A.WayTooLongWords,1,False,False,0.0,0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
2,A.WayTooLongWords,2,False,False,0.0,0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
3,A.WayTooLongWords,7,True,True,1.0,1,0,2.0,2.0,2.0,...,0,0,1,0,0,0,0,0,0,0
4,A.WayTooLongWords,8,True,True,1.0,0,1,1.0,1.0,1.0,...,0,0,0,1,0,0,0,0,0,0


In [39]:
def get_all_blocks(code: str) -> List[Tuple[str, int, int]]:
    block_types = {
        ast.If: "if",
        ast.For: "for",
        ast.While: "while",
        ast.With: "with",
        ast.AsyncWith: "async with",
        ast.AsyncFor: "async for",
        ast.Try: "try",
        ast.FunctionDef: "function",
        ast.AsyncFunctionDef: "async function",
        ast.ClassDef: "class"
    }

    tree = ast.parse(code)
    blocks = []

    def visit(node):
        for child in ast.iter_child_nodes(node):
            for node_type, name in block_types.items():
                if isinstance(child, node_type):
                    start = child.lineno
                    end = getattr(child, 'end_lineno', None)
                    if end is None:
                        # Estimate end if not available (Python <3.8)
                        end = max((getattr(n, 'lineno', start) for n in ast.walk(child) if hasattr(n, 'lineno')), default=start)
                    blocks.append((name, start, end))
            visit(child)

    visit(tree)
    return blocks

In [41]:
file_path = 'solutions/python/SportsRank.py'

with open(file_path, 'r') as file:
    file_content = file.read()

In [42]:
blocks = get_all_blocks(file_content)
for b_type, start, end in blocks:
    print(f"{b_type} block from line {start} to {end}")

function block from line 1 to 15
for block from line 9 to 14
function block from line 16 to 24
for block from line 20 to 23
if block from line 21 to 22
function block from line 25 to 33
for block from line 29 to 32
if block from line 30 to 31
function block from line 34 to 39
for block from line 36 to 38
if block from line 37 to 38
function block from line 40 to 49
for block from line 43 to 47
for block from line 48 to 49
if block from line 50 to 51


In [82]:
selected_problems = pd.read_csv('datasets/selected_problems.csv')
titles_selected_problems = list(selected_problems['title'])

In [135]:
final_crafted_features['blocks'] = [[] for _ in range(len(final_crafted_features))]

In [136]:
final_crafted_features.head()

,title,line_row,any_fix_available,all_fixes_available,fix_availability_ratio,unsafe_fixes,safe_fixes,avg_fix_complexity,max_fix_complexity,min_fix_complexity,...,Q000,RET504,RET505,RET507,RET508,RUF059,SIM110,W291,W292,blocks
0,A+BAgain?,7,True,True,1.0,0,1,1.0,1.0,1.0,...,0,0,0,0,0,0,0,0,1,[]
1,A.WayTooLongWords,1,False,False,0.0,0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,[]
2,A.WayTooLongWords,2,False,False,0.0,0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,[]
3,A.WayTooLongWords,7,True,True,1.0,1,0,2.0,2.0,2.0,...,0,1,0,0,0,0,0,0,0,[]
4,A.WayTooLongWords,8,True,True,1.0,0,1,1.0,1.0,1.0,...,0,0,1,0,0,0,0,0,0,[]


In [6]:
def read_python_file(filename):
    file_path = f"solutions/python/{filename.replace(' ', '')}.py"

    with open(file_path, 'r') as file:
        return file.read()

In [43]:
def get_line_blocks(blocks, line):
    line_blocks = []
    for b_type, start, end in blocks:
        if line >= start and line <= end:
            line_blocks.append(b_type)
    return line_blocks
get_line_blocks(blocks, 12)

['function', 'for']

In [137]:
problems_blocks = {}
for problem_title in titles_selected_problems:
    problem_file_content = read_python_file(problem_title)
    blocks = get_all_blocks(problem_file_content)
    problems_blocks[problem_title.replace(' ','')] = blocks
    
for index, row in final_crafted_features.iterrows():
    line_blocks = get_line_blocks(problems_blocks[row['title'].replace(' ','')], row['line_row'])
    final_crafted_features.loc[index, "blocks"].extend(line_blocks)

In [138]:
final_crafted_features.head()

,title,line_row,any_fix_available,all_fixes_available,fix_availability_ratio,unsafe_fixes,safe_fixes,avg_fix_complexity,max_fix_complexity,min_fix_complexity,...,Q000,RET504,RET505,RET507,RET508,RUF059,SIM110,W291,W292,blocks
0,A+BAgain?,7,True,True,1.0,0,1,1.0,1.0,1.0,...,0,0,0,0,0,0,0,0,1,[for]
1,A.WayTooLongWords,1,False,False,0.0,0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,[function]
2,A.WayTooLongWords,2,False,False,0.0,0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,"[function, if]"
3,A.WayTooLongWords,7,True,True,1.0,1,0,2.0,2.0,2.0,...,0,1,0,0,0,0,0,0,0,"[function, if]"
4,A.WayTooLongWords,8,True,True,1.0,0,1,1.0,1.0,1.0,...,0,0,1,0,0,0,0,0,0,"[function, if]"


In [140]:
encoded_blocks = mlb.fit_transform(final_crafted_features['blocks'])
encoded_line_blocks = pd.DataFrame(encoded_blocks, columns=mlb.classes_)
print(encoded_line_blocks.shape)
final_crafted_features = final_crafted_features.drop(columns=['blocks'])
final_crafted_features = pd.concat([final_crafted_features, encoded_line_blocks], axis=1, join='outer')
final_crafted_features.head()

(261, 5)


,title,line_row,any_fix_available,all_fixes_available,fix_availability_ratio,unsafe_fixes,safe_fixes,avg_fix_complexity,max_fix_complexity,min_fix_complexity,...,RET508,RUF059,SIM110,W291,W292,class,for,function,if,while
0,A+BAgain?,7,True,True,1.0,0,1,1.0,1.0,1.0,...,0,0,0,0,1,0,1,0,0,0
1,A.WayTooLongWords,1,False,False,0.0,0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,1,0,0
2,A.WayTooLongWords,2,False,False,0.0,0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,1,1,0
3,A.WayTooLongWords,7,True,True,1.0,1,0,2.0,2.0,2.0,...,0,0,0,0,0,0,0,1,1,0
4,A.WayTooLongWords,8,True,True,1.0,0,1,1.0,1.0,1.0,...,0,0,0,0,0,0,0,1,1,0


In [141]:
final_crafted_features.to_csv('datasets/final_crafted_features_1.csv', index=False)

### Add Line Characters Number 

In [4]:
crafted_features = pd.read_csv('datasets/final_crafted_features_1.csv')
crafted_features['characters_number'] = 0
crafted_features.head()

,title,line_row,any_fix_available,all_fixes_available,fix_availability_ratio,unsafe_fixes,safe_fixes,avg_fix_complexity,max_fix_complexity,min_fix_complexity,...,RUF059,SIM110,W291,W292,class,for,function,if,while,characters_number
0,A+BAgain?,7,True,True,1.0,0,1,1.0,1.0,1.0,...,0,0,0,1,0,1,0,0,0,0
1,A.WayTooLongWords,1,False,False,0.0,0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,1,0,0,0
2,A.WayTooLongWords,2,False,False,0.0,0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,1,1,0,0
3,A.WayTooLongWords,7,True,True,1.0,1,0,2.0,2.0,2.0,...,0,0,0,0,0,0,1,1,0,0
4,A.WayTooLongWords,8,True,True,1.0,0,1,1.0,1.0,1.0,...,0,0,0,0,0,0,1,1,0,0


In [14]:
def get_characters_for_line(title, line):
    python_content = read_python_file(title)
    # print(python_content)
    lines = python_content.split('\n')
    return len(lines[line-1].replace(' ',''))

get_characters_for_line('A.WayTooLongWords', 2)

15

In [15]:
for index, row in crafted_features.iterrows():
    crafted_features.loc[index, "characters_number"] = get_characters_for_line(row['title'], row['line_row'])
    
crafted_features.head()

,title,line_row,any_fix_available,all_fixes_available,fix_availability_ratio,unsafe_fixes,safe_fixes,avg_fix_complexity,max_fix_complexity,min_fix_complexity,...,RUF059,SIM110,W291,W292,class,for,function,if,while,characters_number
0,A+BAgain?,7,True,True,1.0,0,1,1.0,1.0,1.0,...,0,0,0,1,0,1,0,0,0,20
1,A.WayTooLongWords,1,False,False,0.0,0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,1,0,0,20
2,A.WayTooLongWords,2,False,False,0.0,0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,1,1,0,15
3,A.WayTooLongWords,7,True,True,1.0,1,0,2.0,2.0,2.0,...,0,0,0,0,0,0,1,1,0,22
4,A.WayTooLongWords,8,True,True,1.0,0,1,1.0,1.0,1.0,...,0,0,0,0,0,0,1,1,0,5


In [16]:
crafted_features.to_csv('datasets/only_linter_lines_dataset.csv', index=False)

## Add Non-Linter Lines 

In [121]:
only_linter_crafted_features = pd.read_csv('datasets/only_linter_lines_dataset.csv')

In [122]:
only_linter_crafted_features.columns

Index(['title', 'line_row', 'any_fix_available', 'all_fixes_available',
       'fix_availability_ratio', 'unsafe_fixes', 'safe_fixes',
       'avg_fix_complexity', 'max_fix_complexity', 'min_fix_complexity',
       'sum_fix_complexity', 'unique_issue_types', 'issues_per_line', 'A001',
       'ANN001', 'ANN202', 'ANN204', 'ARG001', 'B007', 'B905', 'C901',
       'COM812', 'D101', 'D102', 'D107', 'E226', 'E271', 'E301', 'E302',
       'E305', 'E306', 'E402', 'E501', 'E741', 'F821', 'F841', 'I001', 'N803',
       'N806', 'PERF401', 'PLC0415', 'PLC1901', 'PLR0913', 'PLR0914',
       'PLR0917', 'PLR1730', 'PLR2004', 'PLR6104', 'PLW0603', 'PLW2901',
       'Q000', 'RET504', 'RET505', 'RET507', 'RET508', 'RUF059', 'SIM110',
       'W291', 'W292', 'class', 'for', 'function', 'if', 'while',
       'characters_number'],
      dtype='object')

In [123]:
problems = only_linter_crafted_features['title'].unique()

In [124]:
def get_total_lines(title):
    python_content = read_python_file(title)
    # print(python_content)
    lines = python_content.split('\n')
    return len(lines)

get_total_lines('A.WayTooLongWords')

21

In [125]:
len(only_linter_crafted_features[
    (only_linter_crafted_features['title'] == 'A+BAgain?') &
    (only_linter_crafted_features['line_row'] == 7)
])

1

In [126]:
columns = only_linter_crafted_features.columns
zero_columns = ['fix_availability_ratio', 'unsafe_fixes', 'safe_fixes', 'avg_fix_complexity', 'max_fix_complexity', 'min_fix_complexity',
       'sum_fix_complexity', 'unique_issue_types', 'issues_per_line', 'ANN001', 'ANN202', 'ANN204', 'ARG001', 'B007', 'B905', 'C901',
       'COM812', 'D101', 'D102', 'D107', 'E226', 'E271', 'E301', 'E302',
       'E305', 'E306', 'E402', 'E501', 'E741', 'F821', 'F841', 'I001', 'N803',
       'N806', 'PERF401', 'PLC0415', 'PLC1901', 'PLR0913', 'PLR0914',
       'PLR0917', 'PLR1730', 'PLR2004', 'PLR6104', 'PLW0603', 'PLW2901',
       'Q000', 'RET504', 'RET505', 'RET507', 'RET508', 'RUF059', 'SIM110',
       'W291', 'W292']
false_columns = ['any_fix_available', 'all_fixes_available']
def add_line_to_dataset(title, line_number):
    new_line = {col: None for col in columns}
    new_line['title'] = title
    new_line['line_row'] = line_number
    
    new_line.update({key: 0 for key in zero_columns})
    new_line.update({key: False for key in false_columns})
    blocks = get_all_blocks(read_python_file(title))
    line_blocks = get_line_blocks(blocks, line_number)
    new_line['class'] = 1 if 'class' in line_blocks else 0
    new_line['for'] = 1 if 'for' in line_blocks else 0
    new_line['function'] = 1 if 'function' in line_blocks else 0
    new_line['if'] = 1 if 'if' in line_blocks else 0
    new_line['while'] = 1 if 'while' in line_blocks else 0
    new_line['characters_number'] = get_characters_for_line(title, line_number)
    
    only_linter_crafted_features.loc[len(only_linter_crafted_features)] = new_line

In [127]:
for problem_title in problems:
    lines_number = get_total_lines(problem_title)
    for line_number in range(1, lines_number + 1):
        # print(line_number)
        has_line = len(only_linter_crafted_features[
            (only_linter_crafted_features['title'] == problem_title) &
            (only_linter_crafted_features['line_row'] == line_number)
        ])
        if has_line:
            continue
        
        add_line_to_dataset(problem_title, line_number)

In [128]:
only_linter_crafted_features.shape

(1160, 65)

In [129]:
only_linter_crafted_features = only_linter_crafted_features.sort_values(by=['title', 'line_row'])
only_linter_crafted_features.head()

,title,line_row,any_fix_available,all_fixes_available,fix_availability_ratio,unsafe_fixes,safe_fixes,avg_fix_complexity,max_fix_complexity,min_fix_complexity,...,RUF059,SIM110,W291,W292,class,for,function,if,while,characters_number
261,A+BAgain?,1,False,False,0.0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,14
262,A+BAgain?,2,False,False,0.0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,1,0,0,0,15
263,A+BAgain?,3,False,False,0.0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,1,0,0,0,14
264,A+BAgain?,4,False,False,0.0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,1,0,0,0,16
265,A+BAgain?,5,False,False,0.0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,1,0,0,0,16


In [130]:
only_linter_crafted_features.to_csv('datasets/all_lines_rf_dataset_no_sim.csv', index=False)

## Add Similarity Features

In [131]:
all_crafted_features = pd.read_csv('datasets/all_lines_rf_dataset_no_sim.csv')
lines_probs = pd.read_csv('datasets/all_probabilities.csv')

In [132]:
all_crafted_features['similarity_2'] = 0.0
all_crafted_features['similarity_3'] = 0.0
all_crafted_features['similarity_4'] = 0.0
all_crafted_features['similarity_5'] = 0.0

In [133]:
for index, row in all_crafted_features.iterrows():
    sim_line = lines_probs[
        (lines_probs['title'].str.replace(' ','') == row['title']) &
        (lines_probs['line_number'] == row['line_row'])
    ]

    if len(sim_line) <= 0:
        break
    all_crafted_features.loc[index, "similarity_2"] = list(sim_line["similarity_2"])[0]
    all_crafted_features.loc[index, "similarity_3"] = list(sim_line["similarity_3"])[0]
    all_crafted_features.loc[index, "similarity_4"] = list(sim_line["similarity_4"])[0]
    all_crafted_features.loc[index, "similarity_5"] = list(sim_line["similarity_5"])[0]

In [135]:
all_crafted_features.head()

,title,line_row,any_fix_available,all_fixes_available,fix_availability_ratio,unsafe_fixes,safe_fixes,avg_fix_complexity,max_fix_complexity,min_fix_complexity,...,class,for,function,if,while,characters_number,similarity_2,similarity_3,similarity_4,similarity_5
0,A+BAgain?,1,False,False,0.0,0,0,0.0,0.0,0.0,...,0,0,0,0,0,14,0.898168,0.849335,0.858385,0.862048
1,A+BAgain?,2,False,False,0.0,0,0,0.0,0.0,0.0,...,0,1,0,0,0,15,0.769206,0.735990,0.821829,0.860018
2,A+BAgain?,3,False,False,0.0,0,0,0.0,0.0,0.0,...,0,1,0,0,0,14,0.932981,0.804449,0.845420,0.874074
3,A+BAgain?,4,False,False,0.0,0,0,0.0,0.0,0.0,...,0,1,0,0,0,16,0.772575,0.777754,0.831880,0.856383
4,A+BAgain?,5,False,False,0.0,0,0,0.0,0.0,0.0,...,0,1,0,0,0,16,0.817759,0.815602,0.854043,0.862512


In [136]:
all_crafted_features.to_csv('datasets/all_lines_rf_dataset_with_sim.csv', index=False)